In [ ]:
'''
Variational Autoencoder(VAE) consists of an encoder, a latent space and a decoder. The output of the encoder is
a mean and variance vector. During training the latent representation that is fed to the decoder is sampled usually using the N(mean = W_m(x), variance = W_v(x)).
This is necessary for our variance matrix W_v to learn, because otherwise the model will just always use the mean. By forcing a reasonable variance and 
we create a smooth latent space where latent representations continously produce reasonable outputs. Without that we would have the same output 
as after sampling a undercomplete autoencoder, where small changes in the latent repr would decode into giberish and def. not generalize well. 

The other trick is our loss function which balances the reconstruction loss and the KL divergence between (usually) a normal 
distribution and the distribution of our latent representations. 

Keep in mind that VAEs are stochastic autoencoders thats why all this prior, posterior and probabilistic distribution
stuff is necessary. The reason the KL_divergence is looking so weird is because since a normal distribuition 
like N(mu, sigma^2) = 1 / sqrt(2*pi*sigma^2) * exp(-0.5 * (x - mu)^2 / sigma^2) and therefore we take the log of our 
prior N(0, 1): log p(z) = -0.5 * log(2 pi) - z^2 / 2 and the log of our posterior N(mu, sigma^2): log q(z|x). 
Then we get: KL(q(z|x) || p(z)) = E_q[log q(z|x) - log p(z)] = 
= 0.5 * (mu^2 + sigma^2 - log(sigma^2) - 1)     (Look at loss function of VAE) 
'''

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=64, latent_dim=2):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.mu = nn.Linear(hidden_dim, latent_dim)
        self.logvar = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z)
        return recon, mu, logvar


def vae_loss(recon, x, mu, logvar, beta=0.02):
    recon_loss = F.mse_loss(recon, x, reduction="mean")
    kl_per_point = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    kl_loss = kl_per_point.mean()
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


model = VAE(latent_dim=2)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
model

VAE(
  (encoder): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
  )
  (mu): Linear(in_features=64, out_features=2, bias=True)
  (logvar): Linear(in_features=64, out_features=2, bias=True)
  (decoder): Sequential(
    (0): Linear(in_features=2, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=2, bias=True)
  )
)